In [ ]:
### Geolibraries
import geopandas as gpd
import osmnx as ox
import contextily as ctx


# R5
import r5py
from r5py import TransportNetwork

# General tools
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import pyarrow.parquet as pq

from h3 import h3
from shapely.geometry import Polygon

In [ ]:
users = pd.read_parquet("scratch/data/users_and_stays_3months_oulu.parquet")

In [ ]:
users_work = users[users["is_work"]==1]
users_work.to_parquet("./data/users_and_works_oulu.parquet")

In [ ]:
users = users[users["home_gid9"].notna()]

In [ ]:
users[users["is_home"]==1]

In [ ]:
stays_summary = (
    users
    .groupby("user_id")
    .agg(
        n_stays=("user_id", "count"),
        max_visit_frequency=("frequency_period", "max")
    )
    .reset_index()
)

In [ ]:
stays_summary.describe()

In [ ]:
plt.figure()
plt.hist(stays_summary["n_stays"], bins=30)
plt.xlabel("Number of stays per user")
plt.ylabel("Count")
plt.title("Distribution of number of stays per user")
plt.show()

In [ ]:
plt.figure()
plt.hist(stays_summary["max_visit_frequency"], bins=30)
plt.xlabel("Maximum visit frequency")
plt.ylabel("Count")
plt.title("Distribution of maximum visit frequency per user")
plt.show()

In [ ]:
plt.figure()
plt.hist(stays_summary["n_stays"], bins=100)
plt.yscale("log")
plt.xlabel("Number of stays per user")
plt.ylabel("Count (log scale)")
plt.title("Distribution of number of stays per user (log scale)")
plt.show()

In [ ]:
jobs = gpd.read_parquet("data/job_distribution_from_census_oulu.parquet")

In [ ]:
pois = pd.read_parquet("data/pois_per_hex_new_class_oulu.parquet")

In [ ]:
jobs = jobs.reset_index()

In [ ]:
users = users.merge(
    jobs[["ID", "weighted_tyo"]],
    left_on="stay_gid9",
    right_on="ID",
    how="left"
).drop(columns="ID")

In [ ]:
# Step 1: Group hsk_pois by 'h3_id' and 'category' to get counts
pois_grouped = (
    pois
    .groupby(['h3_id', 'category'])['count']
    .sum()                           # sum the values instead of counting rows
    .unstack(fill_value=0)           # make wide format, categories as columns
    .reset_index()
)


In [ ]:
pois_grouped

In [ ]:
users = users.merge(
    pois_grouped[
        [
            "h3_id",
            "Education",
            "Healthcare and Health",
            "Others / Not sure",
            "Recreational, Outdoors",
            "Shopping, Errands",
            "Social, Cultural",
        ]
    ],
    left_on="stay_gid9",
    right_on="h3_id",
    how="left"
).drop(columns="h3_id")

In [ ]:
users["user_id"].nunique()

In [ ]:
import pyarrow.parquet as pq

cols_needed = [
    "Origin_Hexagon_ID","Destination_Hexagon_ID", "car_co2"
]

table = pq.read_table("scratch/car_co2_6000_oulu.parquet", 
                      columns=cols_needed,
                      use_threads=True)

df_car_co2 = table.to_pandas(types_mapper=pd.ArrowDtype)  # keeps pandas light

In [ ]:
df_car_co2 = df_car_co2.rename(columns={
    "Origin_Hexagon_ID": "from_id",
    "Destination_Hexagon_ID": "to_id",
    "car_co2": "co2_emissions_g"
})

In [ ]:
df_car_co2_sym = df_car_co2.merge(
    df_car_co2,
    left_on=["from_id", "to_id"],
    right_on=["to_id", "from_id"],
    how="left",
    suffixes=("_outbound", "_inbound")
)

In [ ]:
df_car_co2_sym["car_co2_outbound"] = (
    df_car_co2_sym["co2_emissions_g_outbound"]
)

df_car_co2_sym["car_co2_inbound"] = (
    df_car_co2_sym["co2_emissions_g_inbound"]
    .fillna(df_car_co2_sym["co2_emissions_g_outbound"])
)

df_car_co2_sym["car_co2_total"] = (
    df_car_co2_sym["car_co2_outbound"]
    + df_car_co2_sym["car_co2_inbound"]
)


In [ ]:
df_car_co2_final = (
    df_car_co2_sym
    .rename(columns={
        "from_id_outbound": "from_id",
        "to_id_outbound": "to_id"
    })[
        [
            "from_id",
            "to_id",
            "car_co2_outbound",
            "car_co2_inbound",
            "car_co2_total"
        ]
    ]
)


In [ ]:
df_car_co2 = df_car_co2_final.copy()

In [ ]:
df_car_co2

In [ ]:
users = users.merge(
    df_car_co2_final[["car_co2_total", "from_id", "to_id"]],
    left_on=["home_gid9", "stay_gid9"],
    right_on=["from_id", "to_id"],
    how="left"
)

In [ ]:
users[users["is_home"]==1]

In [ ]:
users.isna().sum()

In [ ]:
users.isna().sum()

In [ ]:
users["from_id_exists"] = users["home_gid9"].isin(df_car_co2_final["from_id"])
users["to_id_exists"] = users["stay_gid9"].isin(df_car_co2_final["to_id"])


In [ ]:
pair_index = set(zip(df_car_co2_final["from_id"], df_car_co2_final["to_id"]))

users["pair_exists"] = list(
    zip(users["home_gid9"], users["stay_gid9"])
)

users["pair_exists"] = users["pair_exists"].isin(pair_index)



In [ ]:
nan_car = users[users["car_co2_total"].isna()]

nan_summary = (
    nan_car
    .groupby(["from_id_exists", "to_id_exists", "pair_exists"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

nan_summary


In [ ]:
users_no_home = users[
    (users["from_id_exists"] == False) &
    (users["to_id_exists"] == True)
]

df_nan = users[users["car_co2_total"].isna()]

In [ ]:
users = users.dropna(subset=["car_co2_total"])

In [ ]:
users

In [ ]:
nan_counts_by_hex = (
    df_nan
    .groupby("stay_gid9")
    .size()
    .reset_index(name="count_nan_car_co2_total")
)


In [ ]:
nan_counts_by_hex

In [ ]:

def h3_to_polygon(h3_id):
    return Polygon(h3.h3_to_geo_boundary(h3_id, geo_json=True))

In [ ]:
gdf_nan_hex = gpd.GeoDataFrame(
    nan_counts_by_hex,  # the car NaN counts
    geometry=nan_counts_by_hex["stay_gid9"].apply(h3_to_polygon),
    crs="EPSG:4326"
)


In [ ]:
gdf_nan_hex.explore(
    column="count_nan_car_co2_total",
    cmap="Reds",
    legend=True,
    tooltip=["stay_gid9", "count_nan_car_co2_total"]
)


In [ ]:
users.to_parquet("./data/user_pois_car_1_oulu.parquet")

In [ ]:
users